# Human-AI Collaboration with MOSAIC

MOSAIC is a grid-based **search-and-rescue** task: a building on fire, victims trapped in rooms, lava on the floor, some doors locked.
A human, an AI, or both together must save the victims before time runs out. This notebook builds the task, looks inside it, plays it,
and records the player's **actions and eye gaze**.

It is the **take-home companion** to the MOSAIC tutorial: work through it at your own pace. Where the tutorial changes
`src/experiment/main.py`, the notebook shows the change twice. **On your laptop** gives the edit, to try in the game.
**Here** is a cell that applies the same setting and shows the result.

| In the tutorial | On your laptop | Here |
|---|---|---|
| Part Two: install | the commands in Step 0 | Step 1 checks the environment |
| Part Three: play the mission | `python -m experiment.main` | Step 7 draws the same window, and lets you play in it |
| 4.1 SAR: building, camera, rewards | `build_sar_env(...)` in `main.py` | Steps 2 and 6 |
| 4.2 GUI: flash, time limit | the `gui =` line, the YAML | Step 7 |
| 4.3 LLM: the AI teammate | `llm_client=` | Step 7 |
| 4.4 Sensing: what is recorded | shown live with an eye tracker | Steps 4, 8 and 9 |

Keep `config.yaml` and `lsl_tools.py` in the same folder as this notebook.

## Step 0: Choose your route

**On JupyterHub.** Select the **Python (smc)** kernel and go to Step 1. Nothing to install.

**On your laptop.** Open the setup below, copy each block into a terminal, then come back to Step 1.

<details>
<summary><b>Laptop setup: click to open the commands to copy</b></summary>

**Before you start:** Python 3.10 or 3.11 (`python --version`), Git, a terminal, and a local desktop, because the game opens its own window.

**1. Clone MOSAIC and create the environment.** Use one of the three.

macOS / Linux
```bash
git clone https://github.com/iHuman-Lab/mosaic.git
cd mosaic
python3 -m venv mosaic_env
source mosaic_env/bin/activate
```

Windows PowerShell
```powershell
git clone https://github.com/iHuman-Lab/mosaic.git
cd mosaic
python -m venv mosaic_env
.\mosaic_env\Scripts\Activate.ps1
```

Conda, any platform
```bash
git clone https://github.com/iHuman-Lab/mosaic.git
cd mosaic
conda create -n mosaic_env python=3.11 -y
conda activate mosaic_env
```

Your prompt now begins with `(mosaic_env)`.

**2. Install.** Five commands, from the `mosaic` folder with `mosaic_env` active.
```bash
python -m pip install --upgrade pip
python -m pip install -e .
python -m pip uninstall -y pygame
python -m pip install --force-reinstall "pygame-ce>=2.5.2"
python -c "import mosaic.sar.env, mosaic.gui.main; print('MOSAIC ready')"
```
The last one prints a `pygame-ce` banner, then `MOSAIC ready`.

**3. Run the mission.**

macOS / Linux
```bash
PYTHONPATH=src python -m experiment.main
```

Windows PowerShell
```powershell
$env:PYTHONPATH = "src"
python -m experiment.main
```
Arrow keys turn and walk, `Space` opens a door, `Tab` picks up a key or rescues a victim, `Alt` asks the AI teammate, `F11` toggles fullscreen, `Esc` quits.

**4. Open this notebook.** Save these two files into the `notebooks` folder, replacing the files of the same name:
[02_mosaic_human_ai.ipynb](https://github.com/Bkdogbey/mosaic/raw/smc2026/notebooks/02_mosaic_human_ai.ipynb) and
[lsl_tools.py](https://github.com/Bkdogbey/mosaic/raw/smc2026/notebooks/lsl_tools.py). Then:
```bash
python -m pip install matplotlib scipy pylsl pyxdf notebook
cd notebooks
python -m jupyter notebook 02_mosaic_human_ai.ipynb
```
Start Jupyter from the `notebooks` folder with `mosaic_env` active: the notebook reads `config.yaml` from that folder.
If Jupyter asks for a kernel, choose **Python 3 (ipykernel)**.

**If something fails**

| Symptom | Fix |
|---|---|
| `cannot import name 'DIRECTION_LTR'` | Rerun install lines 3 and 4 (`pygame-ce`) |
| `No module named 'mosaic'` | Activate `mosaic_env`, rerun `pip install -e .` |
| `No module named 'experiment'` | Run from the repo root with `PYTHONPATH=src` |
| `python --version` shows 3.9 or older, or 3.12 and newer | Recreate `mosaic_env` with Python 3.10 or 3.11 |
| No window, or `No available video device` | Run locally, not over SSH |
| `Alt` replies `Import tabulate failed` | Add `provider: dummy` as a top-level line in `configs/experiment.yaml` |

</details>


## Step 1: Check the environment

The notebook draws off-screen, so pygame uses SDL's `dummy` video driver. It must be set **before** pygame is imported.

In [ ]:
import os, sys
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.abspath("../src"))       # the repository's experiment code (src/experiment)

In [ ]:
import importlib

for name in ["numpy", "matplotlib", "scipy", "pygame", "pygame_gui", "minigrid", "gymnasium", "mosaic", "yaml", "pylsl", "pyxdf", "lsl_tools"]:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", None) or getattr(getattr(module, "version", None), "ver", "")
        print(f"ok       {name:11s} {version}")
    except Exception as exc:
        print(f"MISSING  {name:11s} {type(exc).__name__}: {exc}")

REPO = os.path.isdir("../src/experiment") and os.path.isfile("../configs/experiment.yaml")
print("ok       repository  ../src and ../configs found" if REPO else
      "MISSING  repository  ../src or ../configs not found: this notebook needs the whole MOSAIC repository around it (Step 0)")


**Checkpoint.** Every line says `ok`, including `repository`.

## Step 2: The config file and the search-and-rescue environment

MOSAIC is built on **MiniGrid**. `build_sar_env` creates the whole task: a building of `num_rows x num_cols` rooms, each
`room_size` tiles wide. Three **placers** decide what goes where:

| Placer | Decides |
|---|---|
| `VictimPlacer(num_real_victims=n)` | how many victims per room, and where |
| `LavaPlacer(lava_per_room=n)` | where the lava hazards are |
| `LockedRoomPlacer(locked_room_prob=p)` | which doors are locked (their keys are hidden in other rooms) |

`VictimPlacer` places **real victims only**. The mission you played in the tutorial also has **decoys**, look-alike victims that cost a point;
the end of this step builds it.

`env.render()` returns an **image**. Nothing opens a window.

First the **config file**: a short YAML file next to this notebook. The notebook builds a small building from it, so everything runs in
seconds, and Step 6 changes its values one at a time to show what each one does.

In [ ]:
print(open("config.yaml").read())

In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt

from mosaic.sar.env import build_sar_env
from mosaic.sar.placers import VictimPlacer, LavaPlacer, LockedRoomPlacer

import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)
GAME = config["game"]
print("game config:", GAME)

def make_env(game=None, **changes):
    """Build the task from the ``game`` section of the config (optionally with some values changed)."""
    g = {**(game or GAME), **changes}
    return build_sar_env(
        screen_size=g["screen_size"], num_rows=g["num_rows"], num_cols=g["num_cols"],
        room_size=g["room_size"], tile_size=g["tile_size"],
        victim_placer=VictimPlacer(num_real_victims=g["num_real_victims"]),
        lava_placer=LavaPlacer(lava_per_room=g["lava_per_room"]),
        locked_room_placer=LockedRoomPlacer(locked_room_prob=g["locked_room_prob"]),
    )

def reset_level(env, seed):
    """Start level ``seed``. The placers draw from Python's ``random`` module, so seed it together with the environment."""
    random.seed(seed)
    return env.reset(seed=seed)

env = make_env()
obs, info = reset_level(env, 0)
print("mission:", obs["mission"])
print("max steps:", env.max_steps)

plt.figure(figsize=(6, 6)); plt.imshow(env.render()); plt.axis("off"); plt.title("seed 0: what env.render() returns")
plt.show()

A different `seed` builds a different building. The seed makes experiments **reproducible**: everyone who uses seed 0 plays the same level.
`env.reset(seed=0)` on its own is not enough: it fixes the rooms and doors, but MOSAIC's placers put the victims, lava and keys down with
Python's `random` module. `reset_level` seeds both.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, seed in zip(axes, (1, 2, 3)):
    reset_level(env, seed)
    ax.imshow(env.render()); ax.set_title(f"seed {seed}"); ax.axis("off")
plt.show()

**Checkpoint.** Four different levels. The task parameters (rooms, victims, lava, locks) all come from `config.yaml`; change them there and look again.

### The mission you played in the tutorial

`python -m experiment.main` builds a larger mission than the one above: 3 x 3 rooms of 10 tiles, crowded with real victims and decoys.
`make_mission()` builds it here with the same settings as `main.py` (lines 31 to 44). The **Here** cells in Steps 2, 6 and 7 use it.

Lines such as `Timeout during mission generation` or `Sampling rejected` are the level generator trying again. They are not errors.

In [ ]:
from mosaic.core.camera import AgentFOVCamera, EdgeFollowCamera, AgentConeCamera
from experiment.placers import LavaRiskVictimPlacer, SectorSpreadLavaPlacer       # src/experiment/placers.py

with open("../configs/experiment.yaml") as f:
    STUDY = yaml.safe_load(f)["game"]                                             # the counts main.py reads

def make_mission(**changes):
    """The mission of main.py (lines 31-44). ``changes`` are extra or replaced ``build_sar_env`` arguments."""
    settings = dict(
        screen_size=800, num_rows=3, num_cols=3, room_size=10,
        victim_placer=LavaRiskVictimPlacer(num_real_victims=STUDY.get("num_real_victims", 6),
                                           num_fake_victims=STUDY.get("num_fake_victims", 12)),
        lava_placer=SectorSpreadLavaPlacer(lava_per_room=STUDY.get("lava_per_room", 8)),
        locked_room_placer=LockedRoomPlacer(locked_room_prob=0.5),
        camera_strategy=AgentFOVCamera(),
    )
    return build_sar_env(**{**settings, **changes})

MISSION_SEED = 5                     # one level for every comparison below
mission = make_mission()
obs, _ = reset_level(mission, MISSION_SEED)
print("mission:", obs["mission"])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(mission.render()); axes[0].set_title("What the participant sees: the agent's room")
axes[1].imshow(mission.unwrapped.grid.render(16, mission.unwrapped.agent_pos, mission.unwrapped.agent_dir))
axes[1].set_title("The whole building (not shown to the participant)")
for ax in axes: ax.axis("off")
plt.show()

### 4.1 SAR: the camera

The camera sets what the participant can see of the same building.

**On your laptop** (`src/experiment/main.py`, line 44), then relaunch:
```diff
+from mosaic.core.camera import AgentConeCamera
@@ inside build_sar_env(...) @@
-        camera_strategy=AgentFOVCamera(),
+        camera_strategy=AgentConeCamera(),
```
**Here:** one level, one moment, three cameras.

In [ ]:
cameras = {"EdgeFollowCamera: scrolls with the agent": EdgeFollowCamera(),
           "AgentFOVCamera: the whole room (main.py)": AgentFOVCamera(),
           "AgentConeCamera: only what is ahead": AgentConeCamera()}

reset_level(mission, MISSION_SEED)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, camera) in zip(axes, cameras.items()):
    mission.switch_camera(camera)            # the same level, seen through another camera
    ax.imshow(mission.render()); ax.set_title(title); ax.axis("off")
mission.switch_camera(AgentFOVCamera())
plt.show()

## Step 3: Actions: move the rescuer

The rescuer has seven discrete actions:

| Action | Effect |
|---|---|
| `left`, `right` | turn 90 degrees |
| `forward` | move one tile ahead |
| `pickup` | rescue the victim in front |
| `drop`, `toggle`, `done` | drop an item, open a door, finish |

`env.step(action)` returns the Gymnasium tuple `(observation, reward, terminated, truncated, info)`.

In [ ]:
print(list(env.actions))

reset_level(env, 0)
script = [env.actions.right] + [env.actions.forward] * 5 + [env.actions.left] + [env.actions.forward] * 4
frames, rewards = [env.render()], []
for action in script:
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
frames.append(env.render())
print("steps:", len(script), " total reward:", sum(rewards), " info:", info)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, frame, title in zip(axes, frames, ("before", f"after {len(script)} actions")):
    ax.imshow(frame); ax.set_title(title); ax.axis("off")
plt.show()

Rewards come from **events**: a rescued victim is `+1` by default (`RescueRewards` sets the values), a decoy or a dead victim is negative.
The `info["events"]` list says what happened on each step. Step 6 changes what a decoy costs.

**Checkpoint.** The rescuer has moved. Next we look at the numbers behind the picture.

## Step 4: What the agent observes

`env.step` and `env.reset` also return an **observation**: a dictionary with everything a program needs to understand the game:
the agent's position and direction, the whole building as a **grid of integer codes**, how many victims are saved and how much
health each victim has left.
These are the fields the tutorial's 4.4 Sensing slide lists; the study runner streams them at every frame.


In [ ]:
obs, _ = reset_level(env, 0)
for key, value in obs.items():
    shown = value if not isinstance(value, (list, dict)) else f"{type(value).__name__} of length {len(value)}"
    print(f"{key:18s} {shown}")

The `grid` is a table of codes. `mosaic.sar.observations` defines them: `0` empty floor, `1` wall, `4` lava, `5` real victim, `6` decoy (`FAKE_VICTIM`),
`10`-`29` doors (colour, open/locked), `30`+ keys. Let's draw it.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from mosaic.sar import observations as O

grid = np.array(obs["grid"])
category = np.zeros_like(grid)
category[grid == O.EMPTY] = 0
category[grid == O.WALL] = 1
category[grid == O.LAVA] = 2
category[grid == O.VICTIM] = 3
category[grid == O.FAKE_VICTIM] = 4
category[(grid >= O.DOOR_BASE) & (grid < O.KEY_BASE)] = 5
category[grid >= O.KEY_BASE] = 6
names = ["floor", "wall", "lava", "victim", "decoy", "door", "key"]
colors = ["#eeeeee", "#222222", "#ee6a1c", "#2ca02c", "#d2b92b", "#1f77b4", "#9467bd"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(category, cmap=ListedColormap(colors), norm=BoundaryNorm(np.arange(-0.5, 7.5), 7))
ax.plot(obs["agent_x"], obs["agent_y"], "r>", markersize=14, label="rescuer")
for i, (n, c) in enumerate(zip(names, colors)):
    ax.plot([], [], "s", color=c, label=n)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5)); ax.set_title("The same level as a grid of codes")
plt.show()
print("victim health:", obs["victim_health"])

**Checkpoint.** The grid matches the picture from Step 2: walls around rooms, doors between them, victims in green.
`victim_health` is each victim's remaining health; here it stays at `1.0`.

## Step 5: Agents: random vs a scripted rescuer

Because MOSAIC is a Gymnasium environment, any program can play it. We compare two:

- **random**: picks a random action every step.
- **rescuer**: plans a shortest path (breadth-first search) over the grid to the nearest reachable victim, opens unlocked doors on the way,
  and picks the victim up.

In [ ]:
from collections import deque

DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]       # agent_dir 0..3: right, down, left, up

def passable(env, x, y, goal):
    if (x, y) == goal:
        return True
    obj = env.grid.get(x, y)
    if obj is None:
        return True
    if obj.type == "door":
        return not obj.is_locked
    return False                                  # walls, lava, keys, other victims

def plan(env, goal):
    """Shortest path of cells from the agent to ``goal`` (breadth-first search), or None."""
    start = tuple(int(v) for v in env.agent_pos)
    queue, came = deque([start]), {start: None}
    while queue:
        cell = queue.popleft()
        if cell == goal:
            path = []
            while cell != start:
                path.append(cell); cell = came[cell]
            return path[::-1]
        for dx, dy in DIRS:
            nxt = (cell[0] + dx, cell[1] + dy)
            if nxt not in came and 0 <= nxt[0] < env.width and 0 <= nxt[1] < env.height and passable(env, *nxt, goal):
                came[nxt] = cell; queue.append(nxt)
    return None

def rescuer(env, rng):
    """Next action of the scripted rescuer, or None when no victim can be reached."""
    best = None
    for victim in env.get_all_victims():
        path = plan(env, tuple(int(v) for v in victim.cur_pos))
        if path and (best is None or len(path) < len(best)):
            best = path
    if best is None:
        return None
    nx, ny = best[0]
    ax, ay = (int(v) for v in env.agent_pos)
    want = DIRS.index((nx - ax, ny - ay))
    if env.agent_dir != want:                                   # turn towards the next cell
        return env.actions.right if (want - env.agent_dir) % 4 == 1 else env.actions.left
    front = env.grid.get(nx, ny)
    if front is not None and front.type == "door" and not front.is_open:
        return env.actions.toggle
    return env.actions.pickup if len(best) == 1 else env.actions.forward

def play(policy, seed, max_steps=600):
    env = make_env(); reset_level(env, seed)
    rng = np.random.default_rng(seed)
    for step in range(max_steps):
        action = policy(env, rng)
        if action is None:
            break
        _, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    status = env.get_mission_status()
    return {"saved": status["saved_victims"], "of": status["saved_victims"] + status["remaining_victims"], "steps": step + 1}

random_policy = lambda env, rng: env.actions(int(rng.integers(0, 7)))

In [ ]:
import pandas as pd

rows = []
for seed in range(10):
    for name, policy in (("random", random_policy), ("rescuer", rescuer)):
        rows.append({"seed": seed, "agent": name, **play(policy, seed)})
results = pd.DataFrame(rows)
print(results.groupby("agent")[["saved", "steps"]].mean().round(1))

fig, ax = plt.subplots(figsize=(7, 4))
results.pivot(index="seed", columns="agent", values="saved").plot.bar(ax=ax)
ax.set_ylabel("victims saved (of ~8)"); ax.set_title("Same ten levels, two agents")
plt.tight_layout(); plt.show()

**Checkpoint.** The scripted rescuer saves far more victims than the random agent, but usually not **all**: the victims left over are ones it cannot reach, typically behind **locked
doors** (it never looks for the keys) or blocked by lava. A human, or a smarter AI, has to solve those. That is where human-AI collaboration begins.

## Step 6: How the config changes the game

We keep the **same eight seeds** (0 to 7) and change **one config value at a time**. The scripted rescuer plays every level and we measure how many
victims it saves, how many steps it needs, and how many steps the game allows (`max_steps`, which grows with the size of the building).
Because the seeds and the player are the same, the numbers are the same on every run, and a difference between rows comes from the value we changed
(including where it moves the victims, lava and keys).

In [ ]:
import pandas as pd

def play_level(game, seed, max_steps=1500):
    env = make_env(game); reset_level(env, seed)
    rng = np.random.default_rng(seed)
    for step in range(max_steps):
        action = rescuer(env, rng)
        if action is None:
            break
        _, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    status = env.get_mission_status()
    total = status["saved_victims"] + status["remaining_victims"]
    return {"victims": total, "saved": status["saved_victims"], "steps": step + 1, "max_steps": env.max_steps}

def sweep(changes, seeds=range(8)):
    """``changes`` is a list of dicts of config values to override. One result row per dict."""
    rows = []
    for change in changes:
        game = {**GAME, **change}
        res = pd.DataFrame([play_level(game, s) for s in seeds])
        rows.append({**change, "victims": res["victims"].mean(), "saved": res["saved"].mean(),
                     "saved_%": 100 * res["saved"].sum() / res["victims"].sum(), "steps": res["steps"].mean(),
                     "max_steps": res["max_steps"].mean()})
    return pd.DataFrame(rows).round(1)

print("baseline (config.yaml as it is):")
display(sweep([{}]))

### 1. `num_real_victims`: how many people to save

More victims per room means a bigger task: the rescuer needs more steps and the game allows more of them (`max_steps`). The share saved stays about the same.

In [ ]:
victims_table = sweep([{"num_real_victims": v} for v in (1, 2, 4)])
display(victims_table)

### 2. `lava_per_room`: hazards

Lava is impassable for the scripted rescuer, so more lava means longer detours (more steps) and can cut off a victim. Here the effect is small.

In [ ]:
lava_table = sweep([{"lava_per_room": v} for v in (0, 1, 3, 6)])
display(lava_table)

### 3. `locked_room_prob`: locked doors

A locked room needs its key. The scripted rescuer never looks for keys, so the more rooms are locked, the fewer victims it can reach. At a high probability the share saved collapses.

In [ ]:
locked_table = sweep([{"locked_room_prob": v} for v in (0.0, 0.35, 0.8)])
display(locked_table)

### 4. The size of the building: `num_rows` and `num_cols`

A bigger building has more rooms, so more victims, longer walks and a larger time budget (`max_steps`). The player stops after at most 1500 steps.
A building of only 1 x 1 rooms cannot be generated (the level generator never finishes), so the smallest size is 2 x 2.

In [ ]:
size_table = sweep([{"num_rows": n, "num_cols": n} for n in (2, 3)])
display(size_table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (table, key, title) in zip(axes, [(lava_table, "lava_per_room", "Lava"), (locked_table, "locked_room_prob", "Locked doors"),
                                          (victims_table, "num_real_victims", "Victims per room")]):
    ax.bar(table[key].astype(str), table["saved_%"], color="#4c78a8")
    ax.set_xlabel(key); ax.set_ylabel("victims saved (%)"); ax.set_ylim(0, 105); ax.set_title(title)
plt.tight_layout(); plt.show()

### 5. Rewards: what a mistake costs (4.1 SAR)

Rewards are not in the YAML: they are an argument of `build_sar_env`. The default is `+1` for a real victim and `-1` for a decoy.

**On your laptop** (`src/experiment/main.py`), then relaunch and rescue a decoy:
```diff
+from mosaic.sar.actions import RescueAction, RescueRewards
@@ inside build_sar_env(...) @@
         camera_strategy=AgentFOVCamera(),
+        action=RescueAction(
+            rewards=RescueRewards(fake_victim=-5.0)),
```
**Here:** the same level twice. The agent walks to the nearest decoy with Step 5's planner and picks it up.

In [ ]:
from mosaic.sar.actions import RescueAction, RescueRewards

def rescue_nearest_decoy(env):
    """Walk to the nearest reachable decoy and pick it up. Returns (its cell, steps walked, reward, events)."""
    decoys = [(x, y) for x in range(env.width) for y in range(env.height)
              if env.grid.get(x, y) is not None and env.grid.get(x, y).type.startswith("fake_victim")]
    path = min((p for p in (plan(env, cell) for cell in decoys) if p), key=len)
    for i, (nx, ny) in enumerate(path):
        ax, ay = (int(v) for v in env.agent_pos)
        want = DIRS.index((nx - ax, ny - ay))
        while env.agent_dir != want:
            env.step(env.actions.right)
        front = env.grid.get(nx, ny)
        if front is not None and front.type == "door" and not front.is_open:
            env.step(env.actions.toggle)
        _, reward, _, _, info = env.step(env.actions.pickup if i == len(path) - 1 else env.actions.forward)
    return path[-1], len(path), reward, [event["type"] for event in info["events"]]

for label, changes in (("default", {}),
                       ("fake_victim=-5.0", {"action": RescueAction(rewards=RescueRewards(fake_victim=-5.0))})):
    env_r = make_mission(**changes)
    reset_level(env_r, MISSION_SEED)
    cell, steps, reward, events = rescue_nearest_decoy(env_r)
    print(f"{label:17s} decoy at {cell}, a path of {steps} cells -> reward {reward:+.1f}   events: {events}")

**Checkpoint.** One text file controls how hard the task is: how many people there are to save, how much lava and how many locked doors stand
in the way, and how large the building is. Other keys, such as `max_time`, change what the participant sees; the next step shows that.

*Try it:* change a value in `config.yaml`, for example `num_real_victims: 4` or `locked_room_prob: 0.8`, and re-run from Step 2.

## Step 7: The human interface, drawn off-screen

`SAREnvGUI` is the pygame window a participant plays in: the game view on the left, an information panel (victims saved, time left)
and a chat panel for the AI teammate on the right. `gui.user` holds the game state; `gui.user.step(action)` plays one action,
exactly like a key press. We draw the window on pygame's dummy driver and read its surface as an image.

In [ ]:
import pygame
from mosaic.gui.main import SAREnvGUI

def gui_frame(gui):
    """The GUI window as a (height, width, 3) image."""
    return np.asarray(pygame.surfarray.array3d(gui.window)).swapaxes(0, 1)

env = make_env()
gui = SAREnvGUI(env, config=GAME)
gui.reset()
gui.user.obs, _ = reset_level(env, 0)

gui.render(gui.user.get_frame())
print("window size:", gui.window_size, "  game view:", gui.game_size, "px, panel width:", gui.panel_width, "px")
plt.figure(figsize=(12, 8)); plt.imshow(gui_frame(gui)); plt.axis("off"); plt.title("The participant's screen")
plt.show()

Now let the scripted rescuer act through the GUI, as if the participant pressed keys, and look again.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(60):
    action = rescuer(gui.user.env, rng)
    if action is None:
        break
    gui.user.step(action)
    gui.render(gui.user.get_frame())

plt.figure(figsize=(12, 8)); plt.imshow(gui_frame(gui)); plt.axis("off"); plt.title("After 60 actions")
plt.show()
print("rescued so far:", gui.user.env.get_mission_status())

The config also reaches the screen. `max_time` is the minutes on the participant's clock, so changing it changes the **information panel** the participant reads.
**On your laptop:** add `max_time: 2` as a top-level line of `configs/experiment.yaml`, then relaunch. **Here:**

In [ ]:
panels = {}
for minutes in (5, 1):
    env_m = make_env()
    gui_m = SAREnvGUI(env_m, config={**GAME, "max_time": minutes})
    gui_m.reset(); gui_m.render(gui_m.user.get_frame())
    panels[minutes] = gui_frame(gui_m)[: gui_m.game_size // 2, gui_m.game_size:]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (minutes, img) in zip(axes, panels.items()):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"max_time: {minutes}")
plt.tight_layout(); plt.show()

### 4.2 GUI: the feedback flash

Rescuing a decoy flashes the edge of the game view red. One style per event sets the colour, strength, length and fade.

**On your laptop** (`src/experiment/main.py`), then relaunch and rescue a decoy:
```diff
+from mosaic.gui.feedback import EdgeVignette, VignetteStyle
@@ further down @@
-    gui = SAREnvGUI(env, config=config, llm_client=llm_client)
+    gui = SAREnvGUI(env, config=config, llm_client=llm_client,
+        vignette=EdgeVignette(800, styles={"wrong_victim":
+            VignetteStyle((200, 0, 0), 200, 1.5, "fade")}))
```
**Here:** the same frame, 0.3 s after the same mistake, with the default flash and with the stronger one.

In [ ]:
from mosaic.gui.feedback import EdgeVignette, VignetteStyle

class StoppedClock:
    """A clock we set by hand, to look at the flash at one chosen moment."""
    ms = 0
    def __call__(self):
        return self.ms

gui_f = SAREnvGUI(make_mission(), config={"fullscreen": False})
gui_f.reset()
gui_f.user.obs, _ = reset_level(gui_f.user.env, MISSION_SEED)
frame = gui_f.user.get_frame()
stronger = {"wrong_victim": VignetteStyle((200, 0, 0), 200, 1.5, "fade")}

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, (title, styles) in zip(axes, (("default flash", None), ("stronger and longer", stronger))):
    clock = StoppedClock()
    gui_f.vignette = EdgeVignette(gui_f.game_size, clock_ms=clock, styles=styles)
    gui_f.vignette.trigger([{"type": "wrong_victim"}])        # the event a decoy rescue sends
    clock.ms = 300
    gui_f.render(frame)
    ax.imshow(gui_frame(gui_f)[:, :gui_f.game_size]); ax.set_title(title); ax.axis("off")
plt.show()

### 4.3 LLM: the AI teammate

A teammate is any object with `query(prompt) -> str`. `main.py` starts with a placeholder. The tutorial swaps in `ReliableTeammate`, which reads
the map and points at the nearest reachable victim, or, with probability `1 - reliability`, at the nearest decoy. It needs no API key.

**On your laptop:** save [advisor.py](https://github.com/Bkdogbey/mosaic-smc-tutorial/blob/smc2026/presentation/labs/advisor.py) as
`src/experiment/advisor.py`, edit `src/experiment/main.py`, then relaunch and press `Alt`:
```diff
+import json
+from .advisor import ReliableTeammate
@@ further down @@
-    gui = SAREnvGUI(env, config=config, llm_client=llm_client)
+    gui = SAREnvGUI(env, config=config,
+                    llm_client=ReliableTeammate(0.7),
+                    prompt_builder=json.dumps)
```
**Here:** the placeholder's answer, then ten answers from `ReliableTeammate`. It is seeded, so this cell prints the same replies every time.

In [ ]:
import json
from mosaic.llm.client import DummyLLMClient, ask

obs, _ = reset_level(mission, MISSION_SEED)
print("placeholder teammate :", ask(obs, DummyLLMClient()))

try:
    from experiment.advisor import ReliableTeammate
except ImportError:
    ReliableTeammate = None
    print("\nadvisor.py is not in src/experiment/, so the second teammate is skipped. To add it, save\n"
          "  https://github.com/Bkdogbey/mosaic-smc-tutorial/blob/smc2026/presentation/labs/advisor.py\n"
          "as src/experiment/advisor.py and run this cell again.")

if ReliableTeammate is not None:
    teammate = ReliableTeammate(0.7, seed=2026)
    replies = [ask(obs, teammate, prompt_builder=json.dumps) for _ in range(10)]
    print("ReliableTeammate(0.7), in this run:")
    for honest, target in ((True, "a real victim"), (False, "a decoy")):
        said = [reply for reply, advice in zip(replies, teammate.log) if advice["honest"] == honest]
        print(f"  {len(said)} of {len(replies)} replies pointed at {target}:", f'"{said[0]}"' if said else "")

`0.7` is a probability, not a count: another seed gives a different split. The replies read the same either way, which is the point:
the participant cannot tell good advice from bad without checking.

### Play it here

This is the real MOSAIC window, with the changes from this step and Step 6 switched on: a decoy costs 5 points, the flash is stronger, the clock
is two minutes, and the teammate reads the map. Type one or more keys and press Enter; the window redraws after each entry.

| Type | Does | In the game window |
|---|---|---|
| `w` | walk forward | up arrow |
| `a`, `d` | turn left, right | left and right arrows |
| `o` | open a door | `Space` |
| `p` | pick up a key, or rescue the victim in front | `Tab` |
| `t` | ask the AI teammate | `Alt` |

`wwd` walks two tiles and turns right. Press Enter on an empty line to stop. On a laptop the same game runs in its own window, with the real keys: see Step 0.

In [ ]:
import time
from IPython.display import display, clear_output
from PIL import Image

KEYS = {"w": pygame.K_UP, "a": pygame.K_LEFT, "d": pygame.K_RIGHT, "o": pygame.K_SPACE, "p": pygame.K_TAB, "t": pygame.K_LALT}

def play(gui, seed=MISSION_SEED):
    """Play ``gui`` in this cell: every typed letter is sent to the window as the key press it stands for."""
    gui.reset()
    gui.user.obs, _ = reset_level(gui.user.env, seed)
    while True:
        gui.render(gui.user.get_frame())
        clear_output(wait=True)
        display(Image.fromarray(gui_frame(gui)))
        try:
            typed = input("w a d = move   o = door   p = pick up / rescue   t = ask the teammate   (Enter alone stops): ").lower()
        except Exception:                      # no keyboard here, for example when the notebook is run unattended
            print("Run this cell by itself to play.")
            break
        if not typed:
            break
        for letter in typed:
            if letter in KEYS:
                gui.handle_user_input(pygame.event.Event(pygame.KEYDOWN, key=KEYS[letter]))
                while gui.user.llm_thread is not None and gui.user.llm_thread.is_alive():
                    time.sleep(0.02)           # wait for the teammate's reply
        if gui.user.last_info.get("events"):
            time.sleep(0.3)                    # a rescue just happened: let its flash build up before the redraw
    if gui.user.total_steps == 0:
        return
    status = gui.user.env.get_mission_status()
    print(f"rescued {status['saved_victims']} of {status['saved_victims'] + status['remaining_victims']}, "
          f"total reward {gui.user.total_reward:+.1f}, {gui.user.total_steps} steps")

teammate = {"llm_client": ReliableTeammate(0.7), "prompt_builder": json.dumps} if ReliableTeammate is not None else {}
play(SAREnvGUI(
    make_mission(action=RescueAction(rewards=RescueRewards(fake_victim=-5.0))),     # 4.1: a decoy costs 5 points
    config={"fullscreen": False, "max_time": 2},                                    # 4.2: a two-minute clock
    vignette=EdgeVignette(800, styles=stronger),                                    # 4.2: the stronger flash
    **teammate,                                                                     # 4.3: the map-reading teammate
))

**Checkpoint.** You played the mission with four settings changed. The panel on the right counts rescued victims and shows the clock, and the clock follows the config. This is the screen where a participant's gaze goes.

## Step 8: Record game state and eye gaze to XDF

The recording has three parts:

1. A **state stream** from the game: one marker per action (`agent_x`, `agent_y`, `step_count`, `saved_victims`, `action`, `reward`),
   plus a marker whenever a victim is rescued.
2. A **gaze stream** from the eye tracker. Here it is synthetic: made-up gaze, so no hardware is needed.
3. A **recorder** that writes both to one XDF file.

With a real eye tracker, as shown live in the tutorial, MOSAIC records the same two things, and Step 9 reads that file too.

### Areas of interest (AOIs)

We split the screen into the **game view**, the **information panel** and the **chat panel**, and record those rectangles in the file,
so the analysis knows where the participant could look.

In [ ]:
import time
from lsl_tools import MarkerOutlet, SyntheticGaze, Recorder

XDF_PATH = "mosaic_session.xdf"
SESSION_SECONDS = config["session"]["seconds"]      # from config.yaml
ACT_EVERY = config["session"]["act_every"]         # one game action every N frames (10 = about 3 per second)

env = make_env()
gui = SAREnvGUI(env, config=GAME)
gui.reset()
user = gui.user
user.obs, _ = reset_level(env, 0)

W, H = gui.window_size
AOIS = {"game": [0, 0, gui.game_size, gui.game_size],
        "info": [gui.game_size, 0, gui.panel_width, gui.game_size // 2],
        "chat": [gui.game_size, gui.game_size // 2, gui.panel_width, gui.game_size // 2]}
print("AOIs (x, y, width, height):", AOIS)

markers = MarkerOutlet("MOSAIC-State")
USE_SYNTHETIC = config["eye_tracker"]["source"] == "synthetic"
gaze = SyntheticGaze(width=W, height=H).start() if USE_SYNTHETIC else None      # "live": use the Gaze stream already on the network

def look(x, y):
    """Steer the synthetic gaze (does nothing when a real eye tracker is the source)."""
    if gaze is not None:
        gaze.look_at(x, y)
own = {markers.source_id, gaze.source_id if gaze is not None else "tobii_eye_tracker"}      # this session's two streams, not anyone else's
recorder = Recorder(XDF_PATH, stream_types=("Markers", "Gaze"), source_ids=own).start()
print("recording:", recorder.streams)

The loop runs the game in real time at 30 frames per second. The synthetic gaze **scans around the agent** in the game view,
**glances at the information panel right after every rescue** to check the counter, and looks at the clock every few seconds.
That is a stand-in for how a real participant behaves, and it is the pattern the analysis will look for.

In [ ]:
rng = np.random.default_rng(0)
markers.push("session_start", screen=[W, H], aois=AOIS, synthetic_gaze=USE_SYNTHETIC)

glance_until, next_clock_check, frame, start = 0, 5.0, 0, time.time()
last_state = None
while time.time() - start < SESSION_SECONDS:
    now = time.time() - start

    if frame % ACT_EVERY == 0:
        action = rescuer(user.env, rng)
        if action is None:
            break
        user.step(action)
        obs = user.obs
        markers.push("state", step_count=obs["step_count"], agent_x=obs["agent_x"], agent_y=obs["agent_y"],
                     saved_victims=obs["saved_victims"], remaining_victims=obs["remaining_victims"],
                     action=int(action), reward=float(user.last_reward))
        for event in user.last_info.get("events", []):
            markers.push("rescue" if event.get("type") == "victim_rescued" else "event", **event)
            glance_until = now + 0.6                                   # check the counter

    if now < glance_until:
        info = AOIS["info"]; look(info[0] + 0.5 * info[2], info[1] + 0.25 * info[3])
    elif now > next_clock_check:
        info = AOIS["info"]; look(info[0] + 0.5 * info[2], info[1] + 0.8 * info[3])
        if now > next_clock_check + 0.6:
            next_clock_check = now + 5.0
    elif frame % 12 == 0:
        center = AOIS["game"][2] / 2
        look(center + rng.normal(0, 90), center + rng.normal(0, 90))   # scan around the agent

    gui.render(user.get_frame())              # also paces the loop at 30 frames per second
    frame += 1
    if frame == 150:
        plt.imsave("mosaic_frame.png", gui_frame(gui))

markers.push("session_end", saved=user.obs["saved_victims"], frames=frame)
saved_path = recorder.stop()
markers.close()
if gaze is not None:
    gaze.stop()
print(f"{frame} frames, {user.obs['saved_victims']} victims saved")
print("saved", saved_path, f"({os.path.getsize(saved_path) / 1024:.0f} KB)")

**Checkpoint.** An `.xdf` file with two streams: the game (states and rescues) and the gaze.

## Step 9: Load the XDF and analyse gaze against the game

To analyse a recording from a real session instead, set `XDF_PATH` to that file and `SCREEN` to the size of the screen it was played on.

In [ ]:
import pandas as pd
from lsl_tools import load_streams, gaze_table, marker_table, study_tables, gaze_area, detect_fixations, gaze_heatmap

XDF_PATH = "mosaic_session.xdf"            # <- or the file of a real session
SCREEN = (1920, 1080)                      # <- the size of the screen a real session was played on
streams = load_streams(XDF_PATH)
for s in streams.values():
    span = s["ts"][-1] - s["ts"][0]
    print(f"{s['type']:8s} {s['name']:14s} {len(s['ts']):6d} samples over {span:5.1f} s{'   <-- SYNTHETIC data' if s['synthetic'] else ''}")

if "GameState" in streams:                 # a real session, as MOSAIC records it
    gaze_data, rows = study_tables(streams, SCREEN)
else:
    gaze_data, rows = gaze_table(streams["Gaze"]), marker_table(streams["Markers"])
t0 = min(gaze_data["t"][0], rows[0]["t"])
gt = gaze_data["t"] - t0
for r in rows: r["t"] -= t0

session = next(r for r in rows if r.get("event") == "session_start")
SCREEN, AOIS = tuple(session["screen"]), session["aois"]
states = pd.DataFrame([r for r in rows if r.get("event") == "state"])
rescues = [r for r in rows if r.get("event") == "rescue"]
print(f"{len(states)} game states, {len(rescues)} rescues, gaze valid {gaze_data['valid'].mean():.0%}")

### Where did the participant look? Dwell time per area of interest

In [ ]:
def inside(x, y, rect):
    return (x >= rect[0]) & (x < rect[0] + rect[2]) & (y >= rect[1]) & (y < rect[1] + rect[3])

valid = gaze_data["valid"]
x, y = gaze_data["x"], gaze_data["y"]
share = {name: float(inside(x, y, rect)[valid].mean()) for name, rect in AOIS.items()}
print({k: f"{v:.0%}" for k, v in share.items()})

fixations = detect_fixations(gaze_data["t"], x, y, max_dispersion=40, min_duration=0.10)
heat = gaze_heatmap(x, y, shape=(SCREEN[1], SCREEN[0]))
background = plt.imread("mosaic_frame.png") if "Markers" in streams and os.path.exists("mosaic_frame.png") else None

AREA_COLORS = {"game": "#2a78d6", "info": "#eb6834", "chat": "#1baf7a"}     # one colour per area, in every plot below

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(share.keys(), share.values(), width=0.5, color=[AREA_COLORS[name] for name in share]); axes[0].set_ylabel("share of valid gaze"); axes[0].set_title("Dwell time by area")
for ax in axes[1:]:
    if background is not None:
        ax.imshow(background, extent=[0, SCREEN[0], SCREEN[1], 0])
    else:                                  # no screenshot of that session: outline the areas instead
        for rect in AOIS.values():
            ax.add_patch(plt.Rectangle(rect[:2], rect[2], rect[3], fill=False, edgecolor="gray"))
        ax.set_aspect("equal")
    ax.set_xlim(0, SCREEN[0]); ax.set_ylim(SCREEN[1], 0); ax.axis("off")
axes[1].imshow(heat, extent=[0, SCREEN[0], SCREEN[1], 0], cmap="hot", alpha=0.55); axes[1].set_title("Gaze heatmap")
for f in fixations:
    axes[2].scatter(f["x"], f["y"], s=(f["end"] - f["start"]) * 3000, facecolors="none", edgecolors="cyan", linewidths=1.5)
axes[2].set_title(f"{len(fixations)} fixations")
plt.tight_layout(); plt.show()

### Does gaze follow the game events?

After each rescue, does the participant check the information panel? We compare the share of gaze on the panel in the **1.5 s after a rescue**
with the share in all **other** times. The strip under the progress line shows which area the gaze was in at every moment.

In [ ]:
on_info = inside(x, y, AOIS["info"]) & valid
after = np.zeros(len(gt), dtype=bool)
for r in rescues:
    after |= (gt >= r["t"]) & (gt < r["t"] + 1.5)

print(f"gaze on the info panel within 1.5 s after a rescue: {on_info[after].mean():.0%}")
print(f"gaze on the info panel at other times:              {on_info[~after].mean():.0%}")

area = gaze_area(gaze_data, AOIS)             # per gaze sample: "game", "info", "chat", "elsewhere" or "no data"

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(states["t"], states["saved_victims"], drawstyle="steps-post", color="#52514e", linewidth=2)
for r in rescues:
    for ax in axes: ax.axvline(r["t"], color="#898781", linewidth=1)
axes[0].set_ylabel("victims saved"); axes[0].yaxis.get_major_locator().set_params(integer=True); axes[0].set_title("Game progress (vertical lines = rescues) and the area the eyes were in")
for name, color in {**AREA_COLORS, "elsewhere": "#c3c2b7"}.items():
    axes[1].fill_between(gt, 0, 1, where=(area == name), step="post", color=color, linewidth=0, label=name)
axes[1].set_yticks([]); axes[1].set_xlabel("time (s)")
axes[1].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.55), frameon=False, title="gaze area (blank = no valid gaze)")
plt.tight_layout(); plt.show()

### The rescuer's path through the building

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(states["agent_x"], states["agent_y"], c=states["t"], cmap="viridis", s=18)
for r in rescues:
    nearest = states.iloc[(states["t"] - r["t"]).abs().argmin()]
    ax.plot(nearest["agent_x"], nearest["agent_y"], "r*", markersize=16)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_title("Path (colour = time), red stars = rescues")
plt.show()

## Summary

| What we did | Tool |
|---|---|
| Built a search-and-rescue task from a YAML config and looked at it as an image, a grid and numbers | `config.yaml`, `build_sar_env`, MiniGrid / Gymnasium |
| Changed one config value at a time and measured the effect on difficulty | `sweep`, scripted rescuer |
| Compared a random agent and a planning agent | `env.step`, breadth-first search |
| Rebuilt the tutorial's mission and its camera and reward changes | `make_mission`, `switch_camera`, `RescueRewards` |
| Drew the participant's screen off-screen | `SAREnvGUI` on the pygame dummy driver |
| Changed the feedback flash and the AI teammate | `EdgeVignette`, `LLMClient` |
| Played the real window with those changes, inside the notebook | `SAREnvGUI.handle_user_input` |
| Streamed game state and gaze with LSL and saved them as XDF | `pylsl`, `lsl_tools.Recorder` |
| Linked gaze to game events | `pyxdf`, areas of interest, fixations |
| Read the recording of a real session with an eye tracker | `lsl_tools.study_tables` |

The same four steps fit any human-AI study: **simulate, interact, record, analyse.**